In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [2]:
df = pd.read_csv("Iris.csv")
df.head()

,Id,SepalLengthCm,SepalWidthCm,PetalLengthCm,PetalWidthCm,Species
0,1,NaN,NaN,1.4,0.2,Iris-setosa
1,2.1,4.9,3.0,1.4,0.2,NaN
2,l,5.0,3.2,1.3,0.2,Iris-setosa
3,4,4.6,3.1,1.5,0.2,Iris-setosa
4,5,5.0,3.6,1.4,0.2,Iris-setosa


In [3]:
df.isnull().sum()

Id               0
SepalLengthCm    3
SepalWidthCm     6
PetalLengthCm    1
PetalWidthCm     4
Species          3
dtype: int64

In [4]:
df=df.dropna()

In [5]:
df.isnull().sum()

Id               0
SepalLengthCm    0
SepalWidthCm     0
PetalLengthCm    0
PetalWidthCm     0
Species          0
dtype: int64

In [ ]:
# Convert the features to numeric values
features = ['SepalLengthCm', 'SepalWidthCm', 'PetalLengthCm', 'PetalWidthCm']
for col in features:
    df[col] = pd.to_numeric(df[col], errors='coerce')     # coerce converts incompatible input to NaN
df['Species'] = df['Species'].astype('category').cat.codes
df = df.dropna()
df = df[df['Species'] != -1]
df.tail()

,Id,SepalLengthCm,SepalWidthCm,PetalLengthCm,PetalWidthCm,Species
144,145,6.7,3.3,5.7,2.5,2
145,146,6.7,3.0,5.2,2.3,2
146,147,6.3,2.5,5.0,1.9,2
147,148,6.5,3.0,5.2,2.0,2
149,150,5.9,3.0,5.1,1.8,2


In [6]:
from sklearn.preprocessing import LabelEncoder
le=LabelEncoder()
df['Species']=le.fit_transform(df['Species'])

In [7]:
X = df.drop(['Species', 'Id'], axis=1).values
Y = df['Species'].values
type(X)

numpy.ndarray

In [12]:
from sklearn.model_selection import train_test_split
x_train,x_test,y_train,y_test=train_test_split(X,Y,test_size=0.2)

In [13]:
classes = np.unique(y_train)

In [14]:
# Store the parameters
mean = dict()
variance = dict()
prior = dict()

for c in classes:
  x_c = x_train[y_train == c]
  mean[c] = np.mean(x_c, axis=0)
  variance[c] = np.var(x_c, axis=0)
  prior[c] = len(x_c) / len(x_train)

In [20]:
def gaussian_pdf(x, mean, variance):
  eps = 1e-7    
  coeff = 1 / np.sqrt(2 * np.pi * variance + eps)
  exp = np.exp(-(x-mean)**2 / (2 * variance + eps))
  return coeff * exp

In [16]:
def predict(X):
  predictions = []
  for x in X:
    posteriors = []
    for c in classes:
      prior_log = np.log(prior[c])
      likelihood_log = np.sum(np.log(gaussian_pdf(x, mean[c], variance[c])))
      posteriors.append(prior_log + likelihood_log)
    predictions.append(classes[np.argmax(posteriors)])
  return np.array(predictions)

In [17]:
y_pred = predict(x_test)

In [18]:
labels = np.unique(y_test)
n = len(labels)
print(n)
cm = np.zeros((n, n))
for i in range(len(y_test)):
  actual = np.where(labels == y_test[i])[0][0]
  predicted = np.where(labels == y_pred[i])[0][0]
  cm[actual][predicted] += 1
print(cm)

3
[[10.  0.  0.]
 [ 0.  8.  2.]
 [ 0.  1.  6.]]


In [19]:
total = np.sum(cm)
for i in range(n):    
  TP = cm[i][i]
  FP = np.sum(cm[:, i]) - TP
  FN = np.sum(cm[i, :]) - TP
  TN = total - TP - FP - FN

  accuracy = (TP + TN) / total
  error_rate = 1 - accuracy
  precision = TP / (TP + FP) if (TP+FP) != 0 else 0
  recall = TP / (TP + FN) if (TP+FN) != 0 else 0

  print("Class: ", labels[i])
  print("Accuracy: ", accuracy)
  print("Precision: ", precision)
  print("Recall: ", recall)

Class:  0
Accuracy:  1.0
Precision:  1.0
Recall:  1.0
Class:  1
Accuracy:  0.8888888888888888
Precision:  0.8888888888888888
Recall:  0.8
Class:  2
Accuracy:  0.8888888888888888
Precision:  0.75
Recall:  0.8571428571428571
